# 02 · Base de documentos com citação

O que este notebook mostra:

1. adicionar PDF, Word, planilha, apresentação e texto;
2. buscar por **significado** e por **palavra exata** (códigos como `NF-4521`);
3. cada resultado com a **citação** de onde veio (arquivo + página, slide, aba ou seção);
4. o que acontece ao adicionar o mesmo arquivo de novo, uma versão nova, ou um PDF escaneado.

Os documentos são fictícios e ficam em `eval/fixtures/` (gerados por `make_fixtures.py`).

> **Instalação** (uma vez):
> ```bash
> pip install "pi-memory[local] @ git+https://github.com/Pedro-Bianco/pi-memory"
> ollama pull qwen2.5:14b
> ```
> Todos os dados deste notebook são fictícios. As saídas abaixo vêm de uma execução real.

In [1]:
import json
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # aviso do tqdm no Jupyter

def mostrar(obj, limite=1500):
    """JSON legível (o mesmo formato da biblioteca, do HTTP e do MCP)."""
    texto = json.dumps(obj, ensure_ascii=False, indent=2)
    print(texto if len(texto) <= limite else texto[:limite] + "\n…")

def pasta_limpa(nome):
    """Cada notebook começa do zero: apaga a pasta de dados da execução anterior."""
    shutil.rmtree(nome, ignore_errors=True)
    return nome

In [2]:
from pi_memory import PiMemory

FIXTURES = Path("../../eval/fixtures")
# Só documentos: o modelo de linguagem não é usado aqui, então qualquer função serve.
pm = PiMemory(path=pasta_limpa("dados_02"), llm=lambda prompt: "")

## Adicionar documentos

In [3]:
for nome in ["contrato_exemplo.pdf", "relatorio.docx", "vendas.xlsx", "apresentacao.pptx",
             "notas.md", "manual.txt", "estoque.csv"]:
    r = pm.documents.add(FIXTURES / nome, owner="empresa")
    d = r["document"]
    print(f"{r['status']:<9} {d['source']:<22} {d['format']:<5} {d['chunks']:>3} trechos")

added     contrato_exemplo.pdf   pdf    20 trechos


added     relatorio.docx         docx    1 trechos


added     vendas.xlsx            xlsx    2 trechos


added     apresentacao.pptx      pptx    4 trechos
added     notas.md               md      3 trechos


added     manual.txt             txt     1 trechos


added     estoque.csv            csv     1 trechos


## Perguntar com outras palavras

A pergunta não precisa repetir as palavras do texto. A citação (`citation.label`) diz exatamente
de onde veio cada trecho.

In [4]:
def perguntar(pergunta, limite=3):
    print("?", pergunta)
    for r in pm.documents.search(pergunta, owner="empresa", limit=limite)["results"]:
        print(f"  {r['score']:.3f}  {r['citation']['label']:<38} {r['text'][:70]}")

perguntar("Qual o prazo para entregar os relatórios depois da visita?")
perguntar("Qual a penalidade por pagar a parcela com atraso?")
perguntar("Quantos clientes novos o apresentador quer conquistar até dezembro?")

? Qual o prazo para entregar os relatórios depois da visita?
  1.972  contrato_exemplo.pdf, página 12        Página 12 O prazo de entrega dos relatórios técnicos é de 45 dias corr
  1.556  relatorio.docx, trecho 1               Relatório trimestral da Empresa Exemplo Ltda. As vendas cresceram 12% 
  1.278  notas.md, seção Pagamento              Os fornecedores recebem em 30 dias. A conta de luz vence no dia 15.
? Qual a penalidade por pagar a parcela com atraso?
  2.008  contrato_exemplo.pdf, página 15        Página 15 A multa por atraso no pagamento é de 2% sobre o valor da par
  1.522  relatorio.docx, trecho 1               Relatório trimestral da Empresa Exemplo Ltda. As vendas cresceram 12% 
  1.223  contrato_exemplo.pdf, página 8         Página 8 Os pagamentos são feitos por boleto bancário com vencimento n
? Quantos clientes novos o apresentador quer conquistar até dezembro?
  2.184  apresentacao.pptx, slide 3 (notas)     As notas do orador citam a meta de 120 clientes novos até 

## Palavra exata: códigos, números, nomes

O vetor de significado empata códigos parecidos (`NF-4521` × `NF-4522`); a parte de palavra
exata os separa. O código aparece em dois documentos, e os dois vêm na frente.

In [5]:
perguntar("NF-4521")
perguntar("Arruela 10mm")

? NF-4521
  1.575  vendas.xlsx, aba Vendas, linhas 1–5    Nota: NF-4519 | Cliente: Loja Fictícia SA | Valor: 1200 Nota: NF-4520 
  1.350  contrato_exemplo.pdf, página 7         Página 7 O faturamento é mensal e cada pagamento corresponde à nota fi
  0.536  contrato_exemplo.pdf, página 16        Página 16 Treinamento anual da equipe da contratante está incluído no 
? Arruela 10mm


  1.564  estoque.csv, linhas 2–4                produto: Parafuso M8 | estoque: 1200 | fornecedor: Ferragens Exemplo p
  0.717  contrato_exemplo.pdf, página 10        Página 10 A contratante fornece acesso às áreas técnicas durante o hor
  0.704  contrato_exemplo.pdf, página 8         Página 8 Os pagamentos são feitos por boleto bancário com vencimento n


## Filtros

`filters` aceita igualdade em campos do trecho (`format`, `doc_id`) e em `metadata.<chave>`.

In [6]:
r = pm.documents.search("multa por atraso", owner="empresa", filters={"format": "docx"})
print([x["citation"]["label"] for x in r["results"]])

['relatorio.docx, trecho 1']


## Mesmo arquivo de novo, versão nova, remoção

In [7]:
print(pm.documents.add(FIXTURES / "notas.md", owner="empresa")["status"])   # unchanged

nova = Path("dados_02/notas.md")
nova.write_text("# Notas\n\nA reunião mudou para a sala Verde.", encoding="utf-8")
r = pm.documents.add(nova, owner="empresa")
print(r["status"], r["document"]["chunks"], "trecho")                         # replaced
perguntar("em que sala é a reunião agora?", limite=2)

unchanged


replaced 1 trecho
? em que sala é a reunião agora?
  2.121  notas.md, seção Notas                  A reunião mudou para a sala Verde.
  1.180  relatorio.docx, trecho 1               Relatório trimestral da Empresa Exemplo Ltda. As vendas cresceram 12% 


In [8]:
doc = next(d for d in pm.documents.list(owner="empresa")["documents"] if d["source"] == "manual.txt")
print(pm.documents.remove(doc["id"], owner="empresa"))
print([d["source"] for d in pm.documents.list(owner="empresa")["documents"]])

{'deleted_chunks': 1}
['notas.md', 'estoque.csv', 'apresentacao.pptx', 'vendas.xlsx', 'relatorio.docx', 'contrato_exemplo.pdf']


## PDF escaneado e formato não suportado

Um PDF só de imagens não tem texto: a biblioteca recusa com explicação e não indexa nada.
OCR não é feito nesta versão; um leitor próprio com OCR pode ser passado em
`PiMemory(readers=[...])`.

In [9]:
from pi_memory import PiMemoryError
for nome, dados in [("escaneado.pdf", (FIXTURES / "escaneado.pdf").read_bytes()),
                    ("foto.heic", b"...")]:
    try:
        pm.documents.add_bytes(nome, dados, owner="empresa")
    except PiMemoryError as e:
        print(f"{e.code}: {e.message}")

no_text: Não encontrei texto em 'escaneado.pdf'. Se for um PDF escaneado (imagem), ele precisa de OCR, que esta versão não faz; um leitor próprio com OCR pode ser passado em PiMemory(readers=[...]).

unsupported_format: Formato não suportado (.heic) em 'foto.heic'. Aceitos: .csv, .docx, .markdown, .md, .pdf, .pptx, .txt, .xlsx. Para outro formato, passe um leitor próprio em PiMemory(readers=[...]).


In [10]:
pm.close()